################ MD01: NOTEBOOK OVERVIEW ################
# WILLMA Whisper Transcriber V06

This notebook provides a local-PC batch transcription workflow for audio files stored in a folder.

It is organized as separate sections for:

- setup and notebook usage
- dependency handling
- configuration
- audio discovery and validation
- Whisper model loading
- single-file and batch transcription
- transcript formatting and saving
- progress and error reporting
- end-to-end execution
- preview and verification

The workflow is designed for use in Visual Studio Code with Jupyter support.

################ MD02: V06 IMPROVEMENTS ################
## What's new in V06

V06 builds on V05 and introduces **six targeted enhancements** to the audio preprocessing
pipeline, directly inspired by the findings in
*Audio Pre-Processings For Better Results in the Transcription Pipeline* (Medium, 2024,
[link](https://medium.com/@developerjo0517/audio-pre-processings-for-better-results-in-the-transcription-pipeline-bab1e8f63334)).

The article benchmarks **Voice Activity Detection (VAD)** and **vocal/noise separation (UVR/MDX)**
as the two most effective preprocessing steps for reducing Whisper hallucinations.
V06 implements both using **GPU-accelerated, production-grade** libraries available in the
`whisperx-env` conda environment (NVIDIA RTX 3090, 24 GB VRAM).

---

### Assessment: preprocessing steps considered (GPU-enabled environment)

| # | Technique | Library | VRAM req. | WER impact | Selected |
|---|-----------|---------|-----------|------------|----------|
| 1 | **Silero VAD** — ML-based voice activity detection | `faster_whisper.vad` (bundled ONNX) | CPU only / <1 ms per 30 s | ★★★★★ | ✅ |
| 2 | **Demucs vocal separation** — isolate speech from background | `demucs` `htdemucs` | ~3 GB GPU | ★★★★★ | ✅ |
| 3 | DC offset removal | `numpy` | — | ★★★ | ✅ |
| 4 | Low-pass filter (band-pass completion) | `scipy` | — | ★★★ | ✅ |
| 5 | Pre-emphasis filter | `numpy` | — | ★★★ | ✅ |
| 6 | RMS / loudness normalization | `numpy` | — | ★★★ | ✅ |
| 7 | SNR estimation + logging | `numpy` | — | ★★ (diagnostic) | ✅ |
| 8 | UVR-MDX-NET-Inst_HQ_4 | `audio-separator[gpu]` | ~9 GB | ★★★★★ | ❌ replaced by Demucs |
| 9 | Energy-based VAD | `scipy` (RMS threshold) | — | ★★★ | ❌ replaced by Silero VAD |
| 10 | webrtcvad | C extension | — | ★★★ | ❌ adds C dependency |

> **GPU note**: The article excluded Silero VAD replacement and UVR/MDX because of GPU requirements.
> With an RTX 3090 (24 GB VRAM) these are the clear top-2 choices:
> Silero VAD is the article's #1 recommendation (used inside `faster-whisper`);
> Demucs `htdemucs` delivers equivalent WER improvements to UVR-MDX-NET and has a clean Python API
> already installable in the `whisperx-env`.

---

### 1 — Silero VAD (`apply_silero_vad`) — GPU upgrade replaces energy-based VAD
**Article's #1 recommendation, already bundled in `faster-whisper 1.2.1`.**
Uses the Silero VAD ONNX model (loaded once per session via `faster_whisper.vad.get_vad_model()`).
Each audio frame gets a speech-probability score; frames below `vad_threshold` (0.5)
are classified as silence. Silence gaps ≥ `vad_min_silence_ms` (1 000 ms) are stripped and the
remaining speech segments are stitched together, preserving original timestamps for the API.
Directly combats the "♪♪" hallucination loop and word-repetition patterns Whisper produces
from long non-speech passages. Falls back to the original audio if less than 1 second of speech
would remain.
New CONFIG keys: `vad_threshold`, `vad_neg_threshold`, `vad_min_speech_ms`,
`vad_min_silence_ms`, `vad_speech_pad_ms`.

### 2 — Demucs vocal separation (`apply_demucs_separation`) — new GPU step
**Best WER reducer per the article. RTX 3090 (24 GB) makes this the obvious choice.**
Uses Facebook Research's `htdemucs` model (Hybrid Transformer Demucs, the state-of-the-art
open-source music/speech separator). Resamples audio to the model's native 44 100 Hz,
applies the model on the GPU to produce four stems (drums, bass, other, vocals), discards all
non-vocal stems, then resamples the vocals back to the original sample rate.
Applied as the **first** preprocessing step (before the 16 kHz resample) so the model
receives full-bandwidth audio. Demucs is loaded once per session; VRAM is released via
`torch.cuda.empty_cache()` after inference.
New CONFIG keys: `demucs_enabled`, `demucs_model`, `demucs_device`.

### 3 — DC offset removal (`remove_dc_offset`)
Subtracts the waveform mean before filtering. Removes DC bias from recording hardware
that would shift the zero-crossing threshold of the high-pass filter.

### 4 — Low-pass filter / band-pass completion (`apply_lowpass_filter`)
Butterworth LPF at 8 000 Hz. Complements the existing 90 Hz high-pass to form a true
band-pass. Frequencies above 8 kHz add no phoneme information for Dutch speech but
contribute recording noise and mel-spectrogram aliasing.

### 5 — Pre-emphasis filter (`apply_pre_emphasis`)
`y[n] = x[n] − 0.97 × x[n−1]`. Boosts high-frequency spectral energy, compensating for
the natural roll-off of human speech and aligning the energy distribution with Whisper's
training data.

### 6 — RMS normalization (`normalize_rms`)
Scales audio to a fixed RMS target (0.1) before the peak clamp. Ensures all recordings
reach the same perceived loudness regardless of original recording gain.

### 7 — SNR estimation and logging (`estimate_snr_db`)
Rough SNR (dB) estimate logged before and after preprocessing. Gives immediate feedback
on pipeline effectiveness without requiring reference transcripts.

---

### Complete V06 preprocessing pipeline

```
 read audio (original SR) → to_mono → SNR before
     ↓
 [Demucs htdemucs — GPU]        ← NEW (replaces stationary noisereduce as primary denoiser)
     ↓
 resample → 16 kHz
     ↓
 DC offset removal              ← NEW
     ↓
 high-pass 90 Hz  +  low-pass 8 kHz  (band-pass)   ← low-pass NEW
     ↓
 noisereduce stationary          ← V05 (secondary, catches residual after Demucs)
     ↓
 [Silero VAD — ONNX]            ← NEW (replaces energy-based VAD)
     ↓
 pre-emphasis 0.97              ← NEW
     ↓
 RMS normalize → peak normalize ← RMS NEW
     ↓
 SNR after → write cleaned.wav
```

## Environment requirements

Run all cells with the **`whisperx-env`** conda environment selected as the notebook kernel.

- **VS Code**: bottom-right kernel selector → `Python (whisperx-env)`
- **Terminal**: `conda activate whisperx-env`

---

### Required packages — verified 26 May 2026

| Package | Version | Role | Status |
|---------|---------|------|--------|
| `torch` | 2.8.0+cu128 | GPU tensor ops (Demucs) | ✅ installed |
| `torchaudio` | 2.8.0+cu128 | Audio tensor utilities | ✅ installed |
| `faster-whisper` | 1.2.1 | Bundles Silero VAD ONNX model | ✅ installed |
| `onnxruntime` | 1.26.0 | Silero VAD ONNX inference | ✅ installed |
| `demucs` | 4.0.1 | `htdemucs` vocal separation (GPU) | ✅ installed |
| `numpy` | 2.4.4 | Array math, signal processing | ✅ installed |
| `scipy` | 1.17.1 | Butterworth filters, resample | ✅ installed |
| `soundfile` | 0.13.1 | Audio file I/O (read/write WAV) | ✅ installed |
| `pandas` | 2.2.3 | Tabular results and display | ✅ installed |
| `requests` | 2.34.2 | WILLMA API HTTP calls | ✅ installed |
| `urllib3` | 2.7.0 | HTTP retry adapter | ✅ installed |
| `noisereduce` | — | Stationary noise reduction | ⚠️ **NOT installed** (optional) |

> **`noisereduce` is optional.** The stationary noise reduction step (step 5 of the pipeline)
> is silently skipped when the package is absent — all other preprocessing steps still run.
> To enable it: `pip install noisereduce`

---

### GPU requirements

| Component | Requirement | Available |
|-----------|-------------|-----------|
| CUDA toolkit | ≥ 12.0 | CUDA 12.8 ✅ |
| GPU VRAM for Demucs `htdemucs` | ~3 GB | RTX 3090 · 24 GB ✅ |
| Silero VAD | CPU only (ONNX) | — ✅ |

`demucs_device = "cuda"` in CONFIG. The `htdemucs` model stays loaded in VRAM between files
(singleton pattern) and VRAM is released via `torch.cuda.empty_cache()` after each file.
Set `"demucs_enabled": False` for recordings that are already clean or if VRAM is tight.

---

### First-run model downloads

| Model | Size | Cache location |
|-------|------|----------------|
| Demucs `htdemucs` | ~80 MB | `~/.cache/torch/hub/checkpoints/` |
| Silero VAD | bundled in `faster-whisper` package | no download needed |

In [ ]:
################ CELL01: SETUP — PATHS, IMPORTS AND CONFIGURATION ################
from pathlib import Path

# ── Paths & output folders ────────────────────────────────────────────────────────
NOTEBOOK_PATH        = Path(r"D:\OneDrive - Hogeschool Rotterdam\SURF_PILOT\AI_HUB_PILOT\PYTHON\WILLMA_WHISPER_TRANSCRIBER_V06.ipynb")
######## =====> DEFAULT_INPUT_FOLDER = Path(r"D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS")

DEFAULT_INPUT_FOLDER     = Path(r"C:\Users\PROMET02\Downloads\transfer_3588427_files_5e74d3b0")
DEFAULT_OUTPUT_FOLDER    = DEFAULT_INPUT_FOLDER / "transcriber_v06_outputs"
DEFAULT_CLEANED_FOLDER   = DEFAULT_OUTPUT_FOLDER / "cleaned_audio"
DEFAULT_LOG_FOLDER       = DEFAULT_OUTPUT_FOLDER / "logs"
DEFAULT_PROGRESS_FILE    = DEFAULT_LOG_FOLDER    / "batch_progress.json"
DEFAULT_REFERENCE_FOLDER = DEFAULT_OUTPUT_FOLDER / "references"

for _d in [DEFAULT_OUTPUT_FOLDER, DEFAULT_CLEANED_FOLDER, DEFAULT_LOG_FOLDER, DEFAULT_REFERENCE_FOLDER]:
    _d.mkdir(parents=True, exist_ok=True)

# ── Imports ───────────────────────────────────────────────────────────────────────
import io, json, mimetypes, time
import numpy as np                               # V06: enhanced preprocessing
import pandas as pd
import requests
import soundfile as sf
from IPython.display import display
from requests.adapters import HTTPAdapter
from scipy import signal
from urllib3.util.retry import Retry

try:
    import noisereduce as nr
except ImportError:
    nr = None

# ── HTTP session factory ──────────────────────────────────────────────────────────
def create_retry_session(max_retries=None):
    n = max_retries or (CONFIG["max_retries"] if "CONFIG" in globals() else 3)
    session = requests.Session()
    retry = Retry(total=n, connect=n, read=n, backoff_factor=2,
                  status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=frozenset(["GET", "POST"]), raise_on_status=False)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

# ── Configuration ─────────────────────────────────────────────────────────────────
CONFIG = {
    "input_folder":                        DEFAULT_INPUT_FOLDER,
    "output_folder":                       DEFAULT_OUTPUT_FOLDER,
    "cleaned_folder":                      DEFAULT_CLEANED_FOLDER,
    "log_folder":                          DEFAULT_LOG_FOLDER,
    "progress_file":                       DEFAULT_PROGRESS_FILE,
    "reference_folder":                    DEFAULT_REFERENCE_FOLDER,
    "supported_extensions":                {".wav", ".mp3", ".m4a", ".flac"},
    "language":                            "nl",
    "overwrite_outputs":                   False,
    "preprocess_audio":                    True,
    "prefer_cleaned_audio":                True,
    "peak_normalization_target":           0.98,
    "target_sample_rate":                  16000,
    "highpass_hz":                         90,
    "lowpass_hz":                          8000,       # V06: band-pass upper cutoff (Hz)
    "lowpass_enabled":                     True,       # V06: toggle low-pass filter
    "request_timeout":                     300,
    "max_retries":                         3,
    "noise_reduction_strength":            0.8,
    # ── V06: pre-emphasis & RMS (unchanged from CPU version) ─────────────────────
    "pre_emphasis_coeff":                  0.97,       # V06: standard ASR pre-emphasis coefficient
    "pre_emphasis_enabled":                True,       # V06: toggle pre-emphasis filter
    "rms_target":                          0.1,        # V06: RMS normalization target level
    # ── V06 GPU: Silero VAD (replaces energy-based VAD) ──────────────────────────
    # loaded once per session from faster_whisper.vad (ONNX, bundled, CPU <1ms/chunk)
    "vad_enabled":                         True,       # V06: toggle Silero VAD
    "vad_threshold":                       0.5,        # V06: speech probability threshold (0–1)
    "vad_neg_threshold":                   0.35,       # V06: silence probability threshold (0–1)
    "vad_min_speech_ms":                   250,        # V06: min speech island to keep (ms)
    "vad_min_silence_ms":                  1000,       # V06: min silence gap to strip (ms)
    "vad_speech_pad_ms":                   300,        # V06: padding added around speech (ms)
    # ── V06 GPU: Demucs vocal separation (htdemucs, ~3 GB VRAM) ──────────────────
    # loaded once per session; set demucs_enabled=False for already-clean recordings
    "demucs_enabled":                      True,       # V06: toggle Demucs separation
    "demucs_model":                        "htdemucs", # V06: model name
    "demucs_device":                       "cuda",     # V06: "cuda" or "cpu"
    # ── API / diarization (unchanged from V05) ────────────────────────────────────
    "willma_base_url":                     "https://willma.surf.nl/api/v0",
    "willma_api_key":                      "77696c6c6d61-fe59cb65-989c-4df3-a67c-070b150bfd7e",
    "diarization_model":                   "pyannote/speaker-diarization-3.1",
    "preferred_whisper_names":             ["whisper-large-v3", "faster-whisper-large-v3", "whisper-large", "whisper-medium"],
    "min_overlap_seconds":                 0.15,
    "alternative_pause_threshold":         1.2,
    "max_diarization_attempts":            2,
    "sleep_between_attempts":              5,
    "stop_after_consecutive_diar_failures":3,
    "max_audio_mb":                        24,
}

print(f"Input:  {DEFAULT_INPUT_FOLDER}\nOutput: {DEFAULT_OUTPUT_FOLDER}")
display(pd.DataFrame([CONFIG]))

Input:  C:\Users\PROMET02\Downloads\transfer_3588427_files_5e74d3b0
Output: C:\Users\PROMET02\Downloads\transfer_3588427_files_5e74d3b0\transcriber_v04_outputs


,input_folder,output_folder,cleaned_folder,log_folder,progress_file,reference_folder,supported_extensions,language,overwrite_outputs,preprocess_audio,...,preferred_whisper_names,min_overlap_seconds,alternative_pause_threshold,use_api_diarization,fallback_to_alternative_diarization,max_chunk_attempts,max_diarization_attempts,sleep_between_attempts,stop_after_consecutive_diar_failures,max_audio_mb
0,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,"{.m4a, .flac, .mp3, .wav}",nl,False,True,...,"[whisper-large-v3, faster-whisper-large-v3, wh...",0.15,1.2,True,True,3,2,5,3,24


In [ ]:
################ CELL02: OPTIONAL PACKAGE INSTALLATION ################
# Uncomment if packages are missing in the active environment.

# %pip install pandas requests soundfile scipy noisereduce  # core dependencies
# %pip install demucs                                        # V06 GPU: vocal separation (htdemucs)
# faster-whisper is already installed and bundles the Silero VAD ONNX model

In [ ]:
################ CELL02: ALL FUNCTIONS ################

# ── Path and audio discovery ──────────────────────────────────────────────────────
def normalize_path(p):
    return Path(p).expanduser().resolve()

def find_audio_files(folder_path, supported_extensions=None):
    folder = normalize_path(folder_path)
    exts = supported_extensions or CONFIG["supported_extensions"]
    if not folder.exists():
        return []
    return sorted(
        p for p in folder.glob("*")
        if p.is_file() and p.suffix.lower() in exts
    )

def prepare_batch_file_list(folder_path, supported_extensions=None):
    return pd.DataFrame([
        {"file_path": str(p), "file_name": p.name, "extension": p.suffix.lower()}
        for p in find_audio_files(folder_path, supported_extensions)
    ])

# ── Audio validation ──────────────────────────────────────────────────────────────
def validate_audio_file(file_path, supported_extensions=None):
    path = normalize_path(file_path)
    exts = supported_extensions or CONFIG["supported_extensions"]
    r = {
        "file_path": str(path), "exists": path.exists(), "is_file": path.is_file(),
        "supported_extension": path.suffix.lower() in exts,
        "size_bytes": path.stat().st_size if path.exists() and path.is_file() else 0,
        "readable": False, "valid": False, "message": "",
    }
    if not r["exists"]:              r["message"] = "File does not exist."; return r
    if not r["is_file"]:             r["message"] = "Path is not a file."; return r
    if not r["supported_extension"]: r["message"] = "Unsupported audio extension."; return r
    if r["size_bytes"] <= 0:         r["message"] = "File is empty."; return r
    try:
        sf.info(str(path)); r["readable"] = r["valid"] = True; r["message"] = "OK"
    except Exception as exc:
        r["message"] = f"Audio read failed: {exc}"
    return r

def run_preflight_checks(file_paths):
    return pd.DataFrame([validate_audio_file(p) for p in file_paths])

# ── Model catalog ─────────────────────────────────────────────────────────────────
def find_model_by_name_fragment(items, fragments):
    for frag in [(f or "").lower() for f in fragments]:
        for item in items:
            if frag in str(item.get("name", "")).lower():
                return item
    return None

def load_model_catalog(session=None):
    resp = (session or create_retry_session()).get(
        f"{CONFIG['willma_base_url']}/sequences",
        headers={"X-API-KEY": CONFIG["willma_api_key"], "Content-Type": "application/json"},
        timeout=60)
    resp.raise_for_status()
    return resp.json()

def load_whisper_model(session=None, preferred_names=None):
    catalog = load_model_catalog(session=session)
    sel = find_model_by_name_fragment(catalog, preferred_names or CONFIG["preferred_whisper_names"])
    if sel is None:
        sel = next((x for x in catalog if "whisper" in str(x.get("name", "")).lower()), None)
    if sel is None:
        raise ValueError("No Whisper model found in the WILLMA model catalog.")
    CONFIG["selected_whisper_model_name"] = sel.get("name")
    return sel

# ── Transcription ─────────────────────────────────────────────────────────────────
def load_audio_metadata(file_path):
    path = normalize_path(file_path)
    info = sf.info(str(path))
    return {"file_path": str(path), "samplerate": info.samplerate, "channels": info.channels,
            "duration_seconds": round(info.duration, 3), "format": info.format, "subtype": info.subtype}

def overlap_seconds(s_a, e_a, s_b, e_b):
    return max(0.0, min(e_a, e_b) - max(s_a, s_b))

def request_diarization_with_retries(
    session, base_url, api_key, model_name,
    filename, audio_bytes, mime_type, timeout_value, max_attempts, sleep_secs,
):
    last_resp, last_result = None, {}
    for attempt in range(1, max_attempts + 1):
        try:
            last_resp = session.post(
                f"{base_url}/audio/custom-diarization",
                headers={"X-API-KEY": api_key, "Accept": "application/json"},
                data={"model": model_name},
                files={"file": (filename, audio_bytes, mime_type)},
                timeout=timeout_value)
            try:
                last_result = last_resp.json()
            except ValueError:
                last_result = last_resp.text
            if last_resp.status_code < 400 and isinstance(last_result, dict) and not last_result.get("error"):
                return last_resp, last_result
        except Exception as exc:
            last_resp = None
            last_result = {"error": f"Attempt {attempt}: {exc}"}
        if attempt < max_attempts:
            time.sleep(sleep_secs)
    return last_resp, last_result

def _transcribe_bytes(audio_bytes, filename, session, language):
    """POST raw audio bytes to the WILLMA transcription endpoint and return parsed JSON."""
    resp = (session or create_retry_session()).post(
        f"{CONFIG['willma_base_url']}/audio/transcriptions",
        headers={"X-API-KEY": CONFIG["willma_api_key"], "Accept": "application/json"},
        files={"file": (filename, audio_bytes, "audio/wav")},
        data={"model": CONFIG["selected_whisper_model_name"],
              "language": language or CONFIG["language"],
              "stream": "false", "response_format": "verbose_json",
              "timestamp_granularities[]": "segment"},
        timeout=CONFIG["request_timeout"])
    resp.raise_for_status()
    r = resp.json()
    if isinstance(r, dict) and r.get("error"):
        raise RuntimeError(f"WILLMA API error: {r['error']}")
    return r

def _chunk_wav_and_transcribe(path, session, language):
    """Split a large WAV into chunks, transcribe each, and merge with corrected timestamps."""
    import math
    max_bytes = int(CONFIG.get("max_audio_mb", 24) * 1024 * 1024)
    audio, sr = sf.read(str(path), always_2d=False)
    audio = to_mono(audio)
    bytes_per_sec = sr * 2                           # 16-bit mono
    chunk_secs    = max(60, max_bytes // bytes_per_sec)
    chunk_samples = chunk_secs * sr
    n_chunks      = math.ceil(len(audio) / chunk_samples)

    all_segments, text_parts = [], []
    for i in range(n_chunks):
        s0 = i * chunk_samples
        s1 = min(s0 + chunk_samples, len(audio))
        offset = s0 / sr
        buf = io.BytesIO()
        sf.write(buf, audio[s0:s1], sr, subtype="PCM_16", format="WAV")
        log_message(f"{path.name}: chunk {i+1}/{n_chunks} "
                    f"({offset:.1f}s\u2013{s1/sr:.1f}s, {buf.tell()/1024/1024:.1f} MB)")
        r = _transcribe_bytes(buf.getvalue(), path.name, session, language)
        text_parts.append(r.get("text", "").strip())
        for seg in r.get("segments", []):
            adj = dict(seg)
            adj["start"] = round(float(seg.get("start", 0)) + offset, 3)
            adj["end"]   = round(float(seg.get("end",   0)) + offset, 3)
            all_segments.append(adj)

    return {"file_path": str(path), "metadata": load_audio_metadata(path),
            "detected_language": language or CONFIG["language"],
            "text": " ".join(t for t in text_parts if t),
            "segments": all_segments,
            "raw_result": {"chunked": True, "n_chunks": n_chunks}}

def transcribe_single_file(file_path, session=None, language=None):
    path = normalize_path(file_path)
    s    = session or create_retry_session()
    max_bytes = int(CONFIG.get("max_audio_mb", 24) * 1024 * 1024)
    if path.stat().st_size > max_bytes:
        log_message(f"{path.name} is {path.stat().st_size/1024/1024:.1f} MB \u2014 splitting into chunks")
        return _chunk_wav_and_transcribe(path, s, language)
    mime = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    with path.open("rb") as f:
        audio_bytes = f.read()
    r = _transcribe_bytes(audio_bytes, path.name, s, language)
    return {"file_path": str(path), "metadata": load_audio_metadata(path),
            "detected_language": r.get("language", language or CONFIG["language"]),
            "text": r.get("text", "").strip(), "segments": r.get("segments", []), "raw_result": r}

# ── Output formatting ─────────────────────────────────────────────────────────────
def format_srt_timestamp(secs: float) -> str:
    ms = int(round(secs * 1000))
    h, ms = divmod(ms, 3_600_000); m, ms = divmod(ms, 60_000); s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def prepare_output_paths(file_path, output_folder=None):
    stem = normalize_path(file_path).stem
    out = normalize_path(output_folder or CONFIG["output_folder"])
    out.mkdir(parents=True, exist_ok=True)
    return {"txt":  out / f"{stem}.txt",
            "json": out / f"{stem}.segments.json",
            "csv":  out / f"{stem}.speaker.csv",
            "srt":  out / f"{stem}.speaker.srt"}

def format_segments_as_table(segments):
    return pd.DataFrame(segments) if segments else pd.DataFrame(
        columns=["start", "end", "speaker", "text", "speaker_source", "speaker_overlap"])

def write_txt_transcript(p, text):  Path(p).write_text(str(text or "").strip() + "\n", encoding="utf-8")
def write_json(p, payload):         Path(p).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
def write_csv(p, segments):         format_segments_as_table(segments).to_csv(p, index=False, encoding="utf-8-sig")

def write_srt(output_path, segments):
    blocks = []; idx = 1
    for seg in segments:
        text = str(seg.get("text", "")).strip()
        if not text:
            continue
        blocks.append(f"{idx}\n{format_srt_timestamp(float(seg.get('start', 0)))} --> "
                      f"{format_srt_timestamp(float(seg.get('end', 0)))}\n"
                      f"[{seg.get('speaker', 'UNKNOWN')}] {text}\n")
        idx += 1
    Path(output_path).write_text("\n".join(blocks), encoding="utf-8")

def save_transcription_outputs(result, output_folder=None):
    paths = prepare_output_paths(result["file_path"], output_folder=output_folder)
    write_txt_transcript(paths["txt"], result.get("text", ""))
    write_json(paths["json"], result)
    write_csv(paths["csv"], result.get("segments", []))
    write_srt(paths["srt"],  result.get("segments", []))
    return paths

# ── Audio preprocessing — base helpers ───────────────────────────────────────────
def to_mono(a):
    return a.astype("float32") if getattr(a, "ndim", 1) == 1 else a.mean(axis=1).astype("float32")

def normalize_peak(a, target=0.98):
    peak = float(abs(a).max()) if len(a) else 0.0
    return a if peak <= 0 else ((a / peak) * target).astype("float32")

# ── V06: signal processing helpers (unchanged from CPU version) ───────────────────

def remove_dc_offset(audio):
    """Subtract mean to eliminate DC bias; improves filter stability (V06)."""
    return (audio - float(np.mean(audio))).astype("float32")

def apply_lowpass_filter(audio, sr, cutoff_hz=None):
    """Butterworth low-pass; removes content above speech range (V06)."""
    cutoff = cutoff_hz if cutoff_hz is not None else CONFIG.get("lowpass_hz", 8000)
    freq = min(float(cutoff) / (sr / 2.0), 0.99)
    b, a = signal.butter(4, freq, btype="lowpass")
    return signal.filtfilt(b, a, audio).astype("float32")

def apply_pre_emphasis(audio, coeff=None):
    """Pre-emphasis: y[n] = x[n] - coeff*x[n-1]. Boosts high-freq for ASR (V06)."""
    c = coeff if coeff is not None else CONFIG.get("pre_emphasis_coeff", 0.97)
    if c <= 0.0:
        return audio
    return np.append(audio[0], audio[1:] - c * audio[:-1]).astype("float32")

def normalize_rms(audio, target_rms=None):
    """Scale audio to a consistent RMS level; clips to [-1, 1] after scaling (V06)."""
    tgt = target_rms if target_rms is not None else CONFIG.get("rms_target", 0.1)
    rms = float(np.sqrt(np.mean(audio ** 2))) if len(audio) else 0.0
    if rms <= 0.0:
        return audio
    return np.clip(audio * (tgt / rms), -1.0, 1.0).astype("float32")

def estimate_snr_db(audio, sr, noise_secs=1.5):
    """Rough SNR estimate: ratio of full-signal RMS to leading-noise-floor RMS (dB) (V06)."""
    n = min(len(audio), max(64, int(sr * noise_secs)))
    noise_rms  = float(np.sqrt(np.mean(audio[:n] ** 2))) + 1e-10
    signal_rms = float(np.sqrt(np.mean(audio    ** 2))) + 1e-10
    return round(20.0 * np.log10(signal_rms / noise_rms), 1)

# ── V06 GPU: Silero VAD via faster-whisper (replaces energy-based VAD) ────────────

_SILERO_VAD_MODEL = None   # module-level singleton — loaded once per session

def _get_silero_vad():
    """Load (or reuse) the Silero VAD ONNX model bundled with faster-whisper."""
    global _SILERO_VAD_MODEL
    if _SILERO_VAD_MODEL is None:
        from faster_whisper.vad import get_vad_model
        log_message("Loading Silero VAD model (once per session) ...")
        _SILERO_VAD_MODEL = get_vad_model()
        log_message("Silero VAD model ready.")
    return _SILERO_VAD_MODEL

def apply_silero_vad(audio, sr):
    """Silero VAD via faster-whisper's bundled ONNX model (V06 GPU upgrade).

    Requires 16 kHz mono float32 audio (call after the resample step).
    Speech probability is evaluated per frame; frames below vad_threshold are
    classified as silence. Silence gaps >= vad_min_silence_ms are stripped and
    the remaining speech segments are concatenated.
    Returns (vad_audio, stats_dict). Loaded model is reused for the full batch.
    """
    from faster_whisper.vad import VadOptions, get_speech_timestamps, collect_chunks

    if sr != 16000:
        raise ValueError(f"Silero VAD requires 16 kHz audio — got {sr} Hz. Resample first.")

    vad_model = _get_silero_vad()
    opts = VadOptions(
        threshold             = CONFIG.get("vad_threshold",       0.5),
        neg_threshold         = CONFIG.get("vad_neg_threshold",   0.35),
        min_speech_duration_ms= CONFIG.get("vad_min_speech_ms",   250),
        max_speech_duration_s = float("inf"),
        min_silence_duration_ms=CONFIG.get("vad_min_silence_ms",  1000),
        speech_pad_ms         = CONFIG.get("vad_speech_pad_ms",   300),
    )
    timestamps = get_speech_timestamps(audio, vad_model, sampling_rate=sr, vad_options=opts)

    if not timestamps:
        log_message("Silero VAD: no speech detected — keeping original audio", level="WARNING")
        return audio, {
            "vad_kept_ratio":      1.0,
            "vad_segments":        0,
            "original_duration_s": round(len(audio) / sr, 2),
            "vad_duration_s":      round(len(audio) / sr, 2),
        }

    speech_audio = collect_chunks(audio, timestamps)
    return speech_audio.astype("float32"), {
        "vad_kept_ratio":      round(len(speech_audio) / max(len(audio), 1), 3),
        "vad_segments":        len(timestamps),
        "original_duration_s": round(len(audio)        / sr, 2),
        "vad_duration_s":      round(len(speech_audio) / sr, 2),
    }

# ── V06 GPU: Demucs vocal separation (htdemucs, RTX 3090 ~3 GB VRAM) ─────────────

_DEMUCS_MODEL = None   # module-level singleton — loaded once per session

def _get_demucs_model():
    """Load (or reuse) the htdemucs separation model on the configured device."""
    global _DEMUCS_MODEL
    if _DEMUCS_MODEL is None:
        import torch
        from demucs.pretrained import get_model
        model_name = CONFIG.get("demucs_model",  "htdemucs")
        device     = CONFIG.get("demucs_device", "cuda" if torch.cuda.is_available() else "cpu")
        log_message(f"Loading Demucs model '{model_name}' on {device} (once per session) ...")
        _DEMUCS_MODEL = get_model(model_name)
        _DEMUCS_MODEL.to(device)
        _DEMUCS_MODEL.eval()
        log_message(f"Demucs ready — sources: {_DEMUCS_MODEL.sources}, SR: {_DEMUCS_MODEL.samplerate} Hz")
    return _DEMUCS_MODEL

def apply_demucs_separation(audio_np, sr):
    """GPU-accelerated vocal separation using Demucs htdemucs (V06 GPU upgrade).

    Resamples the input to the model's native 44 100 Hz, runs inference on the
    configured device (cuda / cpu), extracts the 'vocals' stem, and resamples
    back to the original sample rate. The GPU cache is emptied after inference
    to avoid CUDA OOM when the next step also uses VRAM.
    """
    import torch
    from demucs.apply import apply_model
    from demucs.audio import convert_audio

    model  = _get_demucs_model()
    device = next(model.parameters()).device

    # (1, samples) → resample to model SR and expand to model's channel count
    wav = torch.from_numpy(audio_np).unsqueeze(0)                        # (1, samples)
    wav = convert_audio(wav, sr, model.samplerate, model.audio_channels) # (C, samples_44k)
    wav = wav.unsqueeze(0).to(device)                                     # (1, C, samples_44k)

    with torch.no_grad():
        sources = apply_model(model, wav, device=device, progress=False)
    # sources: (batch=1, n_sources, C, samples_44k)

    vocals_idx = model.sources.index("vocals")
    vocals_44k = sources[0, vocals_idx].cpu()   # (C, samples_44k)

    # free VRAM before subsequent steps
    del sources, wav
    torch.cuda.empty_cache()

    # back to mono at original sample rate
    vocals_back = convert_audio(vocals_44k, model.samplerate, sr, 1)  # (1, samples)
    return vocals_back.squeeze(0).numpy().astype("float32")

def preprocess_audio(file_path):
    src = normalize_path(file_path)
    out_dir = normalize_path(CONFIG["cleaned_folder"])
    out_dir.mkdir(parents=True, exist_ok=True)
    dst = out_dir / f"{src.stem}.cleaned.wav"
    if dst.exists() and not CONFIG["overwrite_outputs"]:
        log_message(f"{src.name}: cleaned file already exists \u2014 skipping preprocessing")
        return dst
    mb = src.stat().st_size / 1024 / 1024
    log_message(f"{src.name}: reading audio ({mb:.1f} MB)")
    audio, sr = sf.read(str(src), always_2d=False)
    audio = to_mono(audio)
    dur = len(audio) / sr
    log_message(f"{src.name}: {dur/60:.1f} min, {sr} Hz \u2014 starting V06 preprocessing")

    # SNR estimate before cleaning (V06)
    snr_before = estimate_snr_db(audio, sr)
    log_message(f"{src.name}: SNR before \u2248 {snr_before} dB")

    # ── Step 1: Demucs vocal separation (GPU, original SR) ────────────────────────
    if CONFIG.get("demucs_enabled", True):
        log_message(f"{src.name}: Demucs '{CONFIG.get('demucs_model', 'htdemucs')}' "
                    f"on {CONFIG.get('demucs_device', 'cuda')} ...")
        try:
            audio = apply_demucs_separation(audio, sr)
            log_message(f"{src.name}: Demucs done")
        except Exception as exc:
            log_message(f"{src.name}: Demucs failed ({exc}) \u2014 continuing without separation",
                        level="WARNING")

    # ── Step 2: Resample to 16 kHz ────────────────────────────────────────────────
    if sr != CONFIG["target_sample_rate"]:
        log_message(f"{src.name}: resampling {sr} Hz \u2192 {CONFIG['target_sample_rate']} Hz")
        audio = signal.resample(
            audio, int(round(len(audio) * CONFIG["target_sample_rate"] / sr))
        ).astype("float32")
        sr = CONFIG["target_sample_rate"]

    # ── Step 3: DC offset removal ─────────────────────────────────────────────────
    audio = remove_dc_offset(audio)

    # ── Step 4: High-pass filter (rumble removal) ─────────────────────────────────
    log_message(f"{src.name}: band-pass {CONFIG['highpass_hz']}–{CONFIG['lowpass_hz']} Hz")
    b, a = signal.butter(4, min(CONFIG["highpass_hz"] / (sr / 2), 0.99), btype="highpass")
    audio = signal.filtfilt(b, a, audio).astype("float32")

    # ── Step 5: Low-pass filter (band-pass completion) ────────────────────────────
    if CONFIG.get("lowpass_enabled", True):
        audio = apply_lowpass_filter(audio, sr, CONFIG["lowpass_hz"])

    # ── Step 6: Stationary noise reduction (secondary after Demucs) ───────────────
    if nr is not None:
        log_message(f"{src.name}: stationary noise reduction")
        noise = audio[: min(len(audio), int(sr * 1.5))]
        audio = nr.reduce_noise(
            y=audio, sr=sr, y_noise=noise,
            prop_decrease=CONFIG["noise_reduction_strength"],
            stationary=True,
        ).astype("float32")

    # ── Step 7: Silero VAD (strip silence, ML-based) ──────────────────────────────
    if CONFIG.get("vad_enabled", True):
        log_message(f"{src.name}: Silero VAD "
                    f"(thr={CONFIG.get('vad_threshold', 0.5)}, "
                    f"min_silence={CONFIG.get('vad_min_silence_ms', 1000)} ms)")
        try:
            vad_audio, vad_stats = apply_silero_vad(audio, sr)
            if len(vad_audio) >= sr:       # keep only if >= 1 s of speech remains
                audio = vad_audio
                log_message(
                    f"{src.name}: VAD kept {vad_stats['vad_kept_ratio']*100:.1f}% "
                    f"({vad_stats['vad_duration_s']:.1f}s of {vad_stats['original_duration_s']:.1f}s, "
                    f"{vad_stats['vad_segments']} segment(s))"
                )
            else:
                log_message(f"{src.name}: Silero VAD would leave < 1 s \u2014 VAD skipped",
                            level="WARNING")
        except Exception as exc:
            log_message(f"{src.name}: Silero VAD failed ({exc}) \u2014 continuing without VAD",
                        level="WARNING")

    # ── Step 8: Pre-emphasis ──────────────────────────────────────────────────────
    if CONFIG.get("pre_emphasis_enabled", True):
        audio = apply_pre_emphasis(audio, CONFIG["pre_emphasis_coeff"])

    # ── Step 9: RMS normalization ─────────────────────────────────────────────────
    audio = normalize_rms(audio, CONFIG["rms_target"])

    # ── Step 10: Peak normalization ───────────────────────────────────────────────
    audio = normalize_peak(audio, CONFIG["peak_normalization_target"])

    # SNR after cleaning (V06)
    snr_after = estimate_snr_db(audio, sr)
    log_message(
        f"{src.name}: SNR after \u2248 {snr_after} dB "
        f"(improvement: {snr_after - snr_before:+.1f} dB)"
    )

    log_message(f"{src.name}: writing cleaned file \u2192 {dst.name}")
    sf.write(str(dst), audio, sr, subtype="PCM_16")
    log_message(f"{src.name}: preprocessing done")
    return dst

def choose_transcription_source(original_path, cleaned_path=None):
    orig = normalize_path(original_path)
    cln  = normalize_path(cleaned_path) if cleaned_path else orig.with_name(f"{orig.stem}.cleaned.wav")
    return cln if CONFIG["prefer_cleaned_audio"] and cln.exists() else orig

# ── Already-processed check ───────────────────────────────────────────────────────
def is_already_processed(file_path, settings=None):
    """Return True when outputs for this source already exist.
    Checks both original stem (e.g. 250309_0107) and cleaned stem
    (e.g. 250309_0107.cleaned) to handle files produced by older runs."""
    settings = settings or CONFIG
    src = normalize_path(file_path)
    out = normalize_path(settings.get("output_folder") or CONFIG["output_folder"])

    def _outputs_exist(stem):
        return all(
            (out / fname).exists() for fname in (
                f"{stem}.txt",
                f"{stem}.segments.json",
                f"{stem}.speaker.csv",
                f"{stem}.speaker.srt",
            )
        )

    return _outputs_exist(src.stem) or _outputs_exist(f"{src.stem}.cleaned")

# ── Diarization ───────────────────────────────────────────────────────────────────
def build_turns_from_stt(segments, pause_threshold=1.2):
    if not segments:
        return []
    segs = sorted(segments, key=lambda x: (float(x.get("start", 0)), float(x.get("end", 0))))
    cur = {"start": float(segs[0].get("start", 0)), "end": float(segs[0].get("end", 0)),
           "text": (segs[0].get("text") or "").strip()}
    turns = []
    for seg in segs[1:]:
        s, e, t = float(seg.get("start", 0)), float(seg.get("end", 0)), (seg.get("text") or "").strip()
        if s - cur["end"] >= pause_threshold:
            turns.append(cur); cur = {"start": s, "end": e, "text": t}
        else:
            cur["end"] = max(cur["end"], e)
            if t:
                cur["text"] = (cur["text"] + " " + t).strip()
    turns.append(cur)
    return turns

def alternative_diarization_from_stt(segments, pause_threshold=1.2, start_with="001"):
    cycle = ["002", "001"] if start_with == "002" else ["001", "002"]
    return [{"start": round(float(t["start"]), 3), "end": round(float(t["end"]), 3), "speaker": cycle[i % 2]}
            for i, t in enumerate(build_turns_from_stt(segments, pause_threshold))]

def force_two_speaker_labels(segments):
    counts = {}
    for seg in segments:
        k = str(seg.get("speaker", "")).strip()
        if k and k != "UNKNOWN":
            counts[k] = counts.get(k, 0) + 1
    top = [x[0] for x in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:2]]
    mapping = {top[i]: f"{i+1:03d}" for i in range(len(top))}
    forced = []
    for seg in segments:
        upd = dict(seg)
        k = str(upd.get("speaker", "")).strip()
        upd["speaker"] = mapping.get(k, k if k in ("001", "002") else "UNKNOWN") if k and k != "UNKNOWN" else "UNKNOWN"
        forced.append(upd)
    return forced

def assign_unknown_by_neighbors(segments):
    s = [dict(x) for x in segments]
    for i, seg in enumerate(s):
        if seg.get("speaker") != "UNKNOWN":
            continue
        prev = next((s[j]["speaker"] for j in range(i-1, -1, -1) if s[j].get("speaker") not in (None, "", "UNKNOWN")), None)
        nxt  = next((s[j]["speaker"] for j in range(i+1, len(s)) if s[j].get("speaker") not in (None, "", "UNKNOWN")), None)
        if prev and prev == nxt:
            seg["speaker"] = prev
    return s

def merge_speaker_segments(segments, max_gap=1.0):
    merged = []
    for seg in sorted(segments, key=lambda x: (float(x.get("start", 0)), float(x.get("end", 0)))):
        text = seg.get("text", "").strip()
        if not text:
            continue
        if not merged:
            merged.append(dict(seg)); continue
        prev = merged[-1]
        if (prev.get("speaker") == seg.get("speaker")
                and float(seg.get("start", 0)) - float(prev.get("end", 0)) <= max_gap):
            prev["end"] = seg.get("end", prev.get("end"))
            prev["text"] = (prev.get("text", "").strip() + " " + text).strip()
        else:
            merged.append(dict(seg))
    return merged

def diarize_audio_or_segments(file_path, transcript_segments, session=None):
    path = normalize_path(file_path)
    mime = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    _, result = request_diarization_with_retries(
        session or create_retry_session(),
        CONFIG["willma_base_url"], CONFIG["willma_api_key"],
        CONFIG["diarization_model"], path.name, path.read_bytes(), mime,
        CONFIG["request_timeout"], CONFIG["max_diarization_attempts"], CONFIG["sleep_between_attempts"])
    if isinstance(result, dict) and isinstance(result.get("diarization"), list) and result["diarization"]:
        return result["diarization"]
    return alternative_diarization_from_stt(transcript_segments, pause_threshold=CONFIG["alternative_pause_threshold"])

def align_speakers_to_transcript(transcript_segments, diarization_segments):
    aligned = []
    for seg in transcript_segments or []:
        upd = dict(seg)
        s, e = float(upd.get("start", 0)), float(upd.get("end", 0))
        best_sp, best_ov = "UNKNOWN", 0.0
        for d in diarization_segments or []:
            ov = overlap_seconds(s, e, float(d.get("start", 0)), float(d.get("end", 0)))
            if ov > best_ov:
                best_ov = ov
                best_sp = d.get("speaker") or d.get("label") or d.get("id") or "UNKNOWN"
        upd["speaker"] = best_sp if best_ov >= CONFIG["min_overlap_seconds"] else "UNKNOWN"
        upd["speaker_overlap"] = round(best_ov, 3)
        upd["speaker_source"] = "api_or_fallback"
        aligned.append(upd)
    return merge_speaker_segments(assign_unknown_by_neighbors(force_two_speaker_labels(aligned)))

# ── Post-processing and evaluation ────────────────────────────────────────────────
def postprocess_transcript(text):
    c = " ".join(str(text or "").split())
    return (c[0].upper() + c[1:]) if c and c[0].isalpha() else c

def levenshtein_distance(a, b):
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for ta in a:
        row = [prev[0] + 1]
        for j, tb in enumerate(b, 1):
            row.append(min(row[-1] + 1, prev[j] + 1, prev[j-1] + (ta != tb)))
        prev = row
    return prev[-1]

def evaluate_transcript(reference_text, predicted_text):
    ref  = " ".join(str(reference_text  or "").strip().lower().split())
    pred = " ".join(str(predicted_text or "").strip().lower().split())
    rw, pw = ref.split(), pred.split()
    return {
        "wer": 0.0 if not rw  and not pw  else levenshtein_distance(rw,  pw)  / max(len(rw),  1),
        "cer": 0.0 if not ref and not pred else levenshtein_distance(list(ref), list(pred)) / max(len(ref), 1),
    }

# ── Batch processing ──────────────────────────────────────────────────────────────
import sys

def log_message(msg, level="INFO"):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] [{level}] {msg}")
    sys.stdout.flush()

def summarize_batch_results(results):
    return pd.DataFrame([{
        "file_path":           item.get("file_path", ""),
        "status":              item.get("status", "unknown"),
        "detected_language":   item.get("detected_language", ""),
        "text_length":         len(item.get("text", "") or ""),
        "segment_count":       len(item.get("segments", []) or []),
        "transcription_audio": item.get("transcription_audio", ""),
        "txt_output":          (item.get("output_paths") or {}).get("txt", ""),
        "csv_output":          (item.get("output_paths") or {}).get("csv", ""),
        "error":               item.get("error", ""),
    } for item in results])

def resume_incomplete_runs(progress_file):
    p = Path(progress_file)
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else []

def save_progress_log(progress_file, results):
    p = Path(progress_file)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

def process_single_audio_file(file_path, session=None, settings=None):
    settings = settings or CONFIG
    s = session or create_retry_session()
    v = validate_audio_file(file_path, supported_extensions=settings["supported_extensions"])
    if not v["valid"]:
        return {"file_path": str(file_path), "status": "failed", "error": v["message"]}
    src = normalize_path(file_path)

    # ── Skip if all outputs already exist and overwrite is disabled ───────────────
    if not settings.get("overwrite_outputs") and is_already_processed(src, settings):
        out_paths = prepare_output_paths(src, output_folder=settings.get("output_folder"))
        log_message(f"{src.name}: already cleaned and transcribed \u2014 skipping "
                    f"(set overwrite_outputs=True to reprocess)")
        return {
            "file_path": str(src),
            "status": "skipped",
            "transcription_audio": str(src),
            "output_paths": {k: str(v) for k, v in out_paths.items()},
        }

    cleaned = None
    if settings.get("preprocess_audio"):
        try:
            cleaned = preprocess_audio(src)
        except Exception as exc:
            log_message(f"Preprocessing failed for {src.name}: {exc}", level="WARNING")

    tx_src = choose_transcription_source(src, cleaned)
    result = transcribe_single_file(tx_src, session=s, language=settings.get("language"))
    segs = result.get("segments") or []

    if segs:
        merged = align_speakers_to_transcript(segs, diarize_audio_or_segments(tx_src, segs, session=s))
        if merged:
            result["text"] = postprocess_transcript(
                " ".join(x.get("text", "").strip() for x in merged if x.get("text")).strip()
                or result.get("text", ""))
            result["segments"] = merged

    if not result.get("segments"):
        fb = postprocess_transcript(result.get("text", ""))
        result["text"] = fb
        result["segments"] = [{
            "start": 0.0,
            "end": float((result.get("metadata") or {}).get("duration_seconds", 0.0) or 0.0),
            "speaker": "001", "speaker_source": "transcript_fallback",
            "speaker_overlap": 0.0, "text": fb,
        }] if fb else []

    ref = normalize_path(settings["reference_folder"]) / f"{src.stem}.reference.txt"
    result["evaluation"] = (
        evaluate_transcript(ref.read_text(encoding="utf-8"), result["text"]) if ref.exists() else None
    )
    result.update({"status": "ok", "source_audio": str(src),
                   "cleaned_audio": str(cleaned) if cleaned else "",
                   "transcription_audio": str(tx_src)})
    result["file_path"] = str(src)  # pin to original stem so outputs match is_already_processed
    result["output_paths"] = {
        k: str(v) for k, v in save_transcription_outputs(result, output_folder=settings["output_folder"]).items()
    }
    return result

def process_audio_folder(folder_path, session=None, settings=None):
    settings = settings or CONFIG
    s = session or create_retry_session()
    results = []
    for path in find_audio_files(folder_path, supported_extensions=settings["supported_extensions"]):
        log_message(f"Processing {path.name}")
        try:
            results.append(process_single_audio_file(path, session=s, settings=settings))
        except Exception as exc:
            results.append({"file_path": str(path), "status": "failed", "error": str(exc)})
        save_progress_log(settings["progress_file"], results)
    return results

In [4]:
# ── Diagnostic: list all discovered audio files ───────────────────────────────────
found = find_audio_files(CONFIG["input_folder"])
print(f"Found {len(found)} file(s):")
for p in found:
    print(" ", p)


Found 4 file(s):
  C:\Users\PROMET02\Downloads\transfer_3588427_files_5e74d3b0\250309_0107.MP3
  C:\Users\PROMET02\Downloads\transfer_3588427_files_5e74d3b0\250322_0110.MP3
  C:\Users\PROMET02\Downloads\transfer_3588427_files_5e74d3b0\250527_0111.MP3
  C:\Users\PROMET02\Downloads\transfer_3588427_files_5e74d3b0\260513_0131.MP3


In [5]:
################ CELL03: RUN — DISCOVER, TRANSCRIBE AND VERIFY ################

# ── Discover audio files and run preflight checks ─────────────────────────────────
audio_file_df = prepare_batch_file_list(CONFIG["input_folder"])
display(audio_file_df.head(20))

preflight_df = (run_preflight_checks(audio_file_df["file_path"].tolist())
                if not audio_file_df.empty else pd.DataFrame())
display(preflight_df.head(20))

# ── Load Whisper model ────────────────────────────────────────────────────────────
session = create_retry_session()
selected_model = load_whisper_model(session=session)
print(f"Selected model: {selected_model.get('name')}")

# ── Transcribe valid files ────────────────────────────────────────────────────────
valid_files = (preflight_df.loc[preflight_df["valid"], "file_path"].tolist()
               if not preflight_df.empty else [])

if not valid_files:
    display(pd.DataFrame([{
        "status": "no_valid_audio_files",
        "input_folder": str(CONFIG["input_folder"]),
        "message": "No valid audio files found.",
    }]))
else:
    print(f"Processing {len(valid_files)} file(s)...")
    batch_results = []
    for fp in valid_files:
        try:
            batch_results.append(process_single_audio_file(fp, session=session, settings=CONFIG))
        except Exception as exc:
            batch_results.append({"file_path": str(fp), "status": "failed", "error": str(exc)})
        save_progress_log(CONFIG["progress_file"], batch_results)
    display(summarize_batch_results(batch_results))

# ── Verify outputs ─────────────────────────────────────────────────────────────────
txt_files = sorted(Path(CONFIG["output_folder"]).glob("*.txt"))[:20]
if txt_files:
    display(pd.DataFrame([{
        "txt":       str(f),
        "txt_exists":  f.exists(),
        "json_exists": f.with_suffix("").with_suffix(".segments.json").exists(),
        "csv_exists":  f.with_suffix("").with_suffix(".speaker.csv").exists(),
        "txt_bytes":   f.stat().st_size if f.exists() else 0,
        "preview":     f.read_text(encoding="utf-8")[:200] if f.exists() else "",
    } for f in txt_files]))

,file_path,file_name,extension
0,C:\Users\PROMET02\Downloads\transfer_3588427_f...,250309_0107.MP3,.mp3
1,C:\Users\PROMET02\Downloads\transfer_3588427_f...,250322_0110.MP3,.mp3
2,C:\Users\PROMET02\Downloads\transfer_3588427_f...,250527_0111.MP3,.mp3
3,C:\Users\PROMET02\Downloads\transfer_3588427_f...,260513_0131.MP3,.mp3


,file_path,exists,is_file,supported_extension,size_bytes,readable,valid,message
0,C:\Users\PROMET02\Downloads\transfer_3588427_f...,True,True,True,153507495,True,True,OK
1,C:\Users\PROMET02\Downloads\transfer_3588427_f...,True,True,True,128230159,True,True,OK
2,C:\Users\PROMET02\Downloads\transfer_3588427_f...,True,True,True,95687439,True,True,OK
3,C:\Users\PROMET02\Downloads\transfer_3588427_f...,True,True,True,100112373,True,True,OK


Selected model: openai/whisper-large-v3
Processing 4 file(s)...
[2026-05-26 20:28:34] [INFO] 250309_0107.MP3: already cleaned and transcribed — skipping (set overwrite_outputs=True to reprocess)
[2026-05-26 20:28:34] [INFO] 250322_0110.MP3: reading audio (122.3 MB)
[2026-05-26 20:28:54] [INFO] 250322_0110.MP3: 133.6 min, 44100 Hz — starting preprocessing
[2026-05-26 20:28:54] [INFO] 250322_0110.MP3: resampling 44100 Hz → 16000 Hz (this may take a while)
[2026-05-26 20:45:15] [INFO] 250322_0110.MP3: applying high-pass filter
[2026-05-26 20:45:18] [INFO] 250322_0110.MP3: writing cleaned file → 250322_0110.cleaned.wav
[2026-05-26 20:45:20] [INFO] 250322_0110.MP3: preprocessing done
[2026-05-26 20:45:20] [INFO] 250322_0110.cleaned.wav is 244.6 MB — splitting into chunks
[2026-05-26 20:45:22] [INFO] 250322_0110.cleaned.wav: chunk 1/11 (0.0s–786.0s, 24.0 MB)
[2026-05-26 20:46:46] [INFO] 250322_0110.cleaned.wav: chunk 2/11 (786.0s–1572.0s, 24.0 MB)
[2026-05-26 20:46:59] [INFO] 250322_0110.cle

,file_path,status,detected_language,text_length,segment_count,transcription_audio,txt_output,csv_output,error
0,C:\Users\PROMET02\Downloads\transfer_3588427_f...,skipped,,0,0,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,
1,C:\Users\PROMET02\Downloads\transfer_3588427_f...,ok,nl,96189,641,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,
2,C:\Users\PROMET02\Downloads\transfer_3588427_f...,ok,nl,81040,358,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,
3,C:\Users\PROMET02\Downloads\transfer_3588427_f...,ok,nl,81110,289,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,C:\Users\PROMET02\Downloads\transfer_3588427_f...,


,txt,txt_exists,json_exists,csv_exists,txt_bytes,preview
0,C:\Users\PROMET02\Downloads\transfer_3588427_f...,True,False,False,108512,"Ik eindeloos moet stoer in schuiven, wat dan o..."
1,C:\Users\PROMET02\Downloads\transfer_3588427_f...,True,True,True,96357,"Nee, want laten we met elkaar gewoon proberen ..."
2,C:\Users\PROMET02\Downloads\transfer_3588427_f...,True,True,True,81149,Allereerst ben ik heel erg benieuwd naar wat z...
3,C:\Users\PROMET02\Downloads\transfer_3588427_f...,True,True,True,81228,"Ik ben Marlies, 50 jaar, vrouw. Hoe lang zijn ..."
